In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
    getEstacionamientos,
    loadEstacionSinCTC
    
)
from src.api.api import GraylogAPIProcessor
from src.utils.util import loadEstaciones,loadEstacionComercial
from src.api.APIs import getCirculacionesPlanificadas,getCirculacionesComerciales,getPlanificacionCirculacionesTecnicas

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime
)
from src.processor import SitraProcessor, MIEProcessor
from src.api.APIs import getInfoAPIs


In [ ]:
from src.api.APIs import hacerPeticion
import pandas as pd
import json
import regex

day = pd.to_datetime("2026-02-25").strftime("%Y-%m-%d")
HOSTPATH = "http://info.api.pre.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = json.dumps({"day": day})

response = hacerPeticion("POST", HOSTPATH, data=data)

res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

gct = pd.DataFrame(
    [
        (
            lambda day_train, connections, next_conn, prev_conn, next_circ, prev_circ: {
                "NTécnico": el.get("circulationId", {}).get("number"),
                "Fecha": "-".join(str(i) for i in el.get("circulationId", {}).get("launchingDate", []))
                         if el.get("circulationId", {}).get("launchingDate") else None,
                "NComercial": day_train.get("commercialNumber"),
                "Línea": day_train.get("line"),
                "Empresa": day_train.get("company"),
                "Operador": day_train.get("operator"),
                "Tipo": day_train.get("trainType"),
                "esComercial": day_train.get("commercialTrain"),
                "esEspecial": day_train.get("special"),
                "esVirtual": day_train.get("virtual"),
                "Recorrido": day_train.get("journey"),

                # Posterior
                "NTécnicoPosterior": next_circ.get("number"),
                "FechaTrenPosterior": "-".join(str(i) for i in next_circ.get("launchingDate", []))
                                       if next_circ.get("launchingDate") else None,
                "TipoConecciónPosterior": next_conn.get("type"),
                "TiempoConecciónPosterior": next_conn.get("connectionTime"),
                "EnlaceComercialPosterior": next_conn.get("commercialLink"),
                "CódigoEstaciónPosterior": next_conn.get("stationCode"),

                # Previo
                "NTécnicoPrevio": prev_circ.get("number"),
                "FechaTrenPrevio": "-".join(str(i) for i in prev_circ.get("launchingDate", []))
                                    if prev_circ.get("launchingDate") else None,
                "TipoTrenPrevio": prev_conn.get("type"),
                "TiempoConexiónTrenPrevio": prev_conn.get("connectionTime"),
                "EnlaceComercialTrenPrevio": prev_conn.get("commercialLink"),
                "CódigoEstaciónPrevio": prev_conn.get("stationCode"),

                "EnlaceComercialSiguiente": connections.get("nextCommercialLinks"),
                "EnlaceComercialPrevio": connections.get("previousCommercialLinks"),
            }
        )(
            el.get("dayTrain") or {},
            (el.get("dayTrain") or {}).get("connections") or {},
            ((el.get("dayTrain") or {}).get("connections") or {}).get("next") or {},
            ((el.get("dayTrain") or {}).get("connections") or {}).get("previous") or {},
            (((el.get("dayTrain") or {}).get("connections") or {}).get("next") or {}).get("circulationId") or {},
            (((el.get("dayTrain") or {}).get("connections") or {}).get("previous") or {}).get("circulationId") or {},
        )
        for el in res_data
    ]
)

gct["Línea"] = gct["Línea"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)

In [ ]:
gct

In [ ]:
rotaciones = gct[["NTécnico", "Fecha","NComercial","Línea","esComercial","NTécnicoPosterior","FechaTrenPosterior","TipoConecciónPosterior","TiempoConecciónPosterior","EnlaceComercialPosterior","CódigoEstaciónPosterior","NTécnicoPrevio","FechaTrenPrevio","TipoTrenPrevio","TiempoConexiónTrenPrevio","EnlaceComercialTrenPrevio","CódigoEstaciónPrevio"]].copy()

In [ ]:
rotaciones [rotaciones["NTécnico"] == "35801"]

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\enlaces.xlsx")

In [ ]:
df = pd.read_excel(fname)

In [ ]:
df.columns

In [ ]:
renamed_columns = {
    "Técnico":"NTécnico",
    "Comercial":"NComercial",
    "Unnamed: 9":"Origen",
    "Unnamed: 10":"Destino",
    "Técnico.1":"NTécnicoPosterior",
    "Comercial.1":"NComercialPosterior",
    "Unnamed: 13":"OrigenPosterior",
    "Unnamed: 14":"DestinoPosterior",
    "Comercial.2":"NComercialResultante",
    "Nuevo Producto":"ProductoResultante",
    "PRIMER":"CirculaciónPrimerTrenIndividual",
    "SEGUNDO":"CirculaciónSegundoTrenIndividual",
    "del Tren Técnico":"InformaciónAdcionalTrenTécnico",
    "del Tren Comercial":"InformaciónAdcionalTrenComercial",
    "Observaciones":"Observaciones"
}
df.rename(columns=renamed_columns, inplace=True)
    
    
    

In [ ]:
rotaciones_renfe = df[["NTécnico","Origen","Destino","NTécnicoPosterior","OrigenPosterior","DestinoPosterior"]].copy()

In [ ]:
rotaciones_renfe = rotaciones_renfe[~rotaciones_renfe.isna().any(axis=1)].copy()

In [ ]:
rotaciones_renfe["NTécnico"] = rotaciones_renfe["NTécnico"].astype(str).str.split('.').str[0]

In [ ]:
rotaciones_renfe["NTécnicoPosterior"] = rotaciones_renfe["NTécnicoPosterior"].astype(str).str.split('.').str[0]

In [ ]:
rotaciones_renfe

In [ ]:
Enlaces = rotaciones[rotaciones["TipoConecciónPosterior"] == "LINK"].copy()

In [ ]:
# Merge interno (solo los que están en ambos)
ambos = Enlaces.merge(
    rotaciones_renfe[['NTécnico', 'NTécnicoPosterior']], 
    on='NTécnico', 
    how='inner',
    suffixes=('_enlaces', '_renfe')
)

# 1. Coincide
coincide = ambos[
    ambos['NTécnicoPosterior_enlaces'] == ambos['NTécnicoPosterior_renfe']
]

# 2. No coincide
no_coincide = ambos[
    ambos['NTécnicoPosterior_enlaces'] != ambos['NTécnicoPosterior_renfe']
]

# 3. Solo en enlaces
en_enlaces_no_renfe = Enlaces[
    ~Enlaces['NTécnico'].isin(rotaciones_renfe['NTécnico'])
]

# 4. Solo en renfe
en_renfe_no_enlaces = rotaciones_renfe[
    ~rotaciones_renfe['NTécnico'].isin(Enlaces['NTécnico'])
]

# Resumen
print(f"Coinciden: {len(coincide)}")
print(f"No coinciden: {len(no_coincide)}")
print(f"Solo en enlaces: {len(en_enlaces_no_renfe)}")
print(f"Solo en renfe: {len(en_renfe_no_enlaces)}")

In [ ]:
estaciones = loadEstaciones()

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones = pd.concat([estaciones, estaciones_sin_ctc], ignore_index=True)

In [ ]:
estaciones  = estaciones[["Código", "Nombre"]].copy()

In [ ]:
en_renfe_no_enlaces.head()

In [ ]:
Enlaces = Enlaces.merge(estaciones, left_on="CódigoEstaciónPosterior", right_on="Código", how="left")
coincide = coincide.merge(estaciones, left_on="CódigoEstaciónPosterior", right_on="Código", how="left")
no_coincide = no_coincide.merge(estaciones, left_on="CódigoEstaciónPosterior", right_on="Código", how="left")
en_enlaces_no_renfe = en_enlaces_no_renfe.merge(estaciones, left_on="CódigoEstaciónPosterior", right_on="Código", how="left")


In [ ]:
regex = r'\s+(RAM|AV|RC)$'

In [ ]:
Enlaces['Nombre'] = Enlaces['Nombre'].str.replace(regex, '', regex=True)
coincide['Nombre'] = coincide['Nombre'].str.replace(regex, '', regex=True)
no_coincide['Nombre'] = no_coincide['Nombre'].str.replace(regex, '', regex=True)
en_enlaces_no_renfe['Nombre'] = en_enlaces_no_renfe['Nombre'].str.replace(regex, '', regex=True)


In [ ]:
renamed_columns = {"Nombre":"Nombre Estación Posterior"}
Enlaces.rename(columns=renamed_columns, inplace=True)
coincide.rename(columns=renamed_columns, inplace=True)
no_coincide.rename(columns=renamed_columns, inplace=True)
en_enlaces_no_renfe.rename(columns=renamed_columns, inplace=True)

In [ ]:
Enlaces = Enlaces[["NTécnico","Fecha","Línea","esComercial","NTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior","CódigoEstaciónPosterior","Nombre Estación Posterior"]]
coincide = coincide[["NTécnico","Fecha","Línea","esComercial","NTécnicoPosterior_enlaces","TipoConecciónPosterior","TiempoConecciónPosterior","CódigoEstaciónPosterior","Nombre Estación Posterior"]]
no_coincide = no_coincide[["NTécnico","Fecha","Línea","esComercial","NTécnicoPosterior_enlaces","TipoConecciónPosterior","TiempoConecciónPosterior","CódigoEstaciónPosterior","Nombre Estación Posterior"]] 
en_enlaces_no_renfe = en_enlaces_no_renfe[["NTécnico","Fecha","Línea","esComercial","NTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior","CódigoEstaciónPosterior","Nombre Estación Posterior"]]

In [ ]:
gct_pro = getCirculacionesPlanificadas("2026-02-26")

In [ ]:
gct_pro_1 = getPlanificacionCirculacionesTecnicas("2026-02-26")

In [ ]:
comercial = gct_pro_1[["NTécnico","esComercial"]].copy()


In [ ]:
origenes = gct_pro[["NTécnico","Secuencia","Código"]].copy()

In [ ]:
origenes = origenes.sort_values("Secuencia").groupby("NTécnico", as_index=False).first()

In [ ]:
origenes = origenes.merge(
    estaciones,
    on = "Código",
    how = "left"

)

In [ ]:
origenes["Nombre"] =origenes['Nombre'].str.replace(regex, '', regex=True)

In [ ]:
renamed_columns = {
    "NTécnico":"NTécnico_1",
    "Código":"CódigoORigenNTécnicoPosterior",
    "Nombre":"NombreOrigenNTécnicoPosterior"
}

In [ ]:
origenes.rename(columns=renamed_columns,inplace=True)

In [ ]:
renamed_columns = {
    "NTécnico":"NTécnico_1",
    "esComercial":"esPosteriorComercial"
}
comercial.rename(columns=renamed_columns,inplace=True)

In [ ]:
Enlaces = Enlaces.merge(origenes, left_on="NTécnicoPosterior", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1","Secuencia"])
coincide = coincide.merge(origenes, left_on="NTécnicoPosterior_enlaces", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1","Secuencia"])
no_coincide = no_coincide.merge(origenes, left_on="NTécnicoPosterior_enlaces", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1","Secuencia"])
en_enlaces_no_renfe = en_enlaces_no_renfe.merge(origenes, left_on="NTécnicoPosterior", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1","Secuencia"])

In [ ]:
Enlaces = Enlaces.merge(comercial, left_on="NTécnicoPosterior", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1"])
coincide = coincide.merge(comercial, left_on="NTécnicoPosterior_enlaces", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1"])
no_coincide = no_coincide.merge(comercial, left_on="NTécnicoPosterior_enlaces", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1"])
en_enlaces_no_renfe = en_enlaces_no_renfe.merge(comercial, left_on="NTécnicoPosterior", right_on="NTécnico_1", how="left").drop(columns=["NTécnico_1"])

In [ ]:
Enlaces

In [ ]:
Enlaces = Enlaces[["NTécnico","Fecha","Línea","esComercial","CódigoEstaciónPosterior","Nombre Estación Posterior","NTécnicoPosterior","esPosteriorComercial","CódigoORigenNTécnicoPosterior","NombreOrigenNTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior"]].copy()
coincide = coincide[["NTécnico","Fecha","Línea","esComercial","CódigoEstaciónPosterior","Nombre Estación Posterior","NTécnicoPosterior_enlaces","esPosteriorComercial","CódigoORigenNTécnicoPosterior","NombreOrigenNTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior",]].copy()
no_coincide = no_coincide[["NTécnico","Fecha","Línea","esComercial","CódigoEstaciónPosterior","Nombre Estación Posterior","NTécnicoPosterior_enlaces","esPosteriorComercial","CódigoORigenNTécnicoPosterior","NombreOrigenNTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior"]].copy()
en_enlaces_no_renfe = en_enlaces_no_renfe[["NTécnico","Fecha","Línea","esComercial","CódigoEstaciónPosterior","Nombre Estación Posterior","NTécnicoPosterior","esPosteriorComercial","CódigoORigenNTécnicoPosterior","NombreOrigenNTécnicoPosterior","TipoConecciónPosterior","TiempoConecciónPosterior"]].copy()

In [ ]:
Enlaces.drop_duplicates(keep="first",inplace=True)
coincide.drop_duplicates(keep="first",inplace=True)
no_coincide.drop_duplicates(keep="first",inplace=True)
en_enlaces_no_renfe.drop_duplicates(keep="first",inplace=True)

In [ ]:
data ={
    "Enlaces": Enlaces,
    "Coinciden": coincide,
    "NoCoinciden": no_coincide,
    "en_GCT_no_exl": en_enlaces_no_renfe,
    "en_exl_no_GCT": en_renfe_no_enlaces
    
}

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\2026_02_26_Comparativa_Enlaces.xlsx")

In [ ]:
guardarExcelMulti(data, fname)

<h1> GCT VS GCC CIRCULACIONES <H1>

In [ ]:
from pathlib import Path
import pandas as pd
import json

def get_circulaciones_comerciales():
    fname = Path("data/circulaciones_lineas_comerciales.json")
    
    with open(fname, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.json_normalize(
        data["lines"],
        sep="_"
    )
    renamed_columns = {
        "technicalNumber": "NTécnico",
        "startDate": "FechaIncio",
        "endDate": "FechaFin",
        "core": "Núcleo",
        "line": "Línea",
    }
    df.rename(columns=renamed_columns, inplace=True)
    df["FechaIncio"] = pd.to_datetime(df["FechaIncio"], unit="ms")
    df["FechaFin"] = pd.to_datetime(df["FechaFin"], unit="ms")


    return df

In [ ]:
gcc = get_circulaciones_comerciales()

In [ ]:
gcc

In [ ]:
gct = getPlanificacionCirculacionesTecnicas("2026-02-20")

In [ ]:
gct